# Convergence across chains, large network, UCI boston split 0

The large network has hidden layers of 256, 128 and 64 units ($d = 44\,802$). Every PDMP was run as four chains, full batch, with $10^7$ gradient evaluations per chain and $w = 0.05$ for the sticky samplers. Chain $c$ starts from its own MAP (map $c$, Adam from its own random initialisation). The Boomerang samplers use each chain's own Laplace covariance. **NUTS** runs 4 chains of 1000 draws from each of the same four MAPs.

The ZigZag draws are stored in random order without their times, so this notebook compares chains only through their distributions. There are no traces, within-chain ESS or lag checks. R-hat stays valid, since splitting shuffled draws gives two random halves.

Samplers whose four chains are not all finished are left out and listed when loading. Statistics are cached per run file in `_stats_cache.pkl` next to the runs.

In [ ]:
import os
import sys
import warnings
from pathlib import Path

import arviz as az
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, "notebooks")
warnings.filterwarnings("ignore", module="arviz")

from utils.convergence_utils import LARGE, large_paths, cached_draw_stats, nuts_chains
from sazz.gpu_friendly.scripts.uci_bnn_grid import load_raw_datasets, make_split, BASE_SEED

DATASET, SPLIT_ID, W = "boston", 0, 0.05
CACHE = LARGE / f"{DATASET}/split_{SPLIT_ID:02d}" / "_stats_cache.pkl"
PLOT_DIR = Path("results/plots_v2/convergence")
PAPER_DIR = PLOT_DIR / "paper"
PAPER_DIR.mkdir(parents=True, exist_ok=True)

# Same colours and order as uci_convergence.ipynb
SAMPLERS = {
    "zigzag":           ("C2", "ZigZag"),
    "boomerang":        ("C0", "Boomerang"),
    "nuts":             ("C4", "NUTS"),
    "sticky_zigzag":    ("C3", "Sticky ZigZag"),
    "sticky_boomerang": ("C1", "Sticky Boomerang"),
}
STATS = ["log sigma", "train RMSE", "test RMSE", "zero %"]
PAPER_RC = {"font.size": 14, "axes.titlesize": 15, "axes.labelsize": 14, "xtick.labelsize": 12,
            "ytick.labelsize": 12, "legend.fontsize": 12}
plt.rcParams.update({
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
    "figure.dpi": 120, "savefig.dpi": 300, "pdf.fonttype": 42,
})

X_all, y_all = load_raw_datasets((DATASET,))[DATASET]
data = make_split(X_all, y_all, seed=BASE_SEED + SPLIT_ID, dtype=torch.float64, device="cpu")

chain_ids, chains = {}, {}
for s in SAMPLERS:
    found = large_paths(s, DATASET, SPLIT_ID)
    if len(found) < 4:
        print(f"{SAMPLERS[s][1]:<18} {len(found)}/4 chains finished, left out")
        continue
    chain_ids[s] = [c for c, _ in found]
    chains[s] = [cached_draw_stats(p, data, CACHE) for _, p in found]
    print(f"{SAMPLERS[s][1]:<18} chains {chain_ids[s]}")
ORDER = [s for s in SAMPLERS if s in chains]


def label(s):
    return f"{SAMPLERS[s][1]}, $w={W:g}$" if "sticky" in s else SAMPLERS[s][1]

## 1. R-hat

Rank-normalised R-hat (Vehtari et al., 2021) over the four chains. For NUTS, R-hat pools all 16 chains, 4 from each of the 4 MAPs, so like the PDMPs it compares chains from different starting modes.

In [ ]:
def rhat(a):
    return float(az.rhat(a, method="rank"))


def stat_array(s, k):
    """[chains, draws] of statistic k, NUTS split into its 16 chains."""
    if s == "nuts":
        return np.concatenate([nuts_chains(st, k) for st in chains[s]])
    return np.stack([st[k] for st in chains[s]])


rows = []
for s in ORDER:
    row = {"sampler": label(s)}
    for k in STATS:
        if k == "zero %" and "sticky" not in s:
            continue
        row[f"R-hat {k}"] = rhat(stat_array(s, k))
    row["total grads"] = sum(st["grads"] for st in chains[s])
    rows.append(row)
conv_df = pd.DataFrame(rows).set_index("sampler")
rhat_cols = [c for c in conv_df.columns if c.startswith("R-hat")]
display(conv_df.style.format(precision=3, na_rep="").format({"total grads": "{:,.0f}"})
        .background_gradient(cmap="Reds", vmin=1.0, vmax=1.5, subset=rhat_cols))

## 2. The mean of every statistic per chain

Where R-hat is high, this shows how the chains disagree: whether one chain is off or all of them differ.

In [ ]:
rows = []
for s in ORDER:
    ids = chain_ids.get(s, list(range(4))) if s != "nuts" else [0, 1, 2, 3]
    for k in STATS:
        if k == "zero %" and "sticky" not in s:
            continue
        rows.append({"sampler": label(s), "statistic": k,
                     **{f"chain {c}": float(st[k].mean()) for c, st in zip(ids, chains[s])},
                     "spread of chain means / posterior sd": float(np.ptp([st[k].mean() for st in chains[s]])
                                                                   / np.mean([st[k].std() for st in chains[s]]))})
means_df = pd.DataFrame(rows).set_index(["sampler", "statistic"])
means_df.style.format(precision=3, na_rep="")

## 3. Chains against their starting MAP

Chain $c$ of every sampler, and NUTS run $c$, start at MAP $c$. If a sampler forgets where it started, the chains' test RMSE agrees whatever their MAPs scored. If it does not, each chain follows its own MAP.

In [ ]:
rows = []
for s in ORDER:
    ids = [0, 1, 2, 3] if s == "nuts" else chain_ids[s]
    row = {"sampler": label(s)}
    for c, st in zip(ids, chains[s]):
        row[f"MAP {c}"], row[f"chain {c}"] = st["MAP test RMSE"], float(st["test RMSE"].mean())
    row["corr(MAP, chain)"] = float(np.corrcoef([st["MAP test RMSE"] for st in chains[s]],
                                                [st["test RMSE"].mean() for st in chains[s]])[0, 1])
    rows.append(row)
map_df = pd.DataFrame(rows).set_index("sampler")
map_df = map_df[sorted(map_df.columns, key=lambda c: (c.startswith("corr"), int(c.split()[-1]) if c[-1].isdigit() else 0, c))]
display(map_df.style.format(precision=2))

# the starting MAPs themselves
st0 = chains[ORDER[0]]
pd.DataFrame({"MAP test RMSE": [st["MAP test RMSE"] for st in st0], "MAP train RMSE": [st["MAP train RMSE"] for st in st0],
              "MAP log sigma": [st["MAP log sigma"] for st in st0]},
             index=[f"MAP {c}" for c in (chain_ids.get(ORDER[0]) or [0, 1, 2, 3])]).style.format(precision=3)

# Paper material

### Figure, chain means for the large network

In [ ]:
def shades(color, n=4):
    base = np.array(mcolors.to_rgb(color))
    return [tuple(base * (1 - a) + a * np.ones(3)) for a in np.linspace(0.0, 0.55, n)]


SHORT = {"zigzag": "ZigZag", "boomerang": "Boomerang", "nuts": "NUTS",
         "sticky_zigzag": "Sticky\nZigZag", "sticky_boomerang": "Sticky\nBoomerang"}
with plt.rc_context(PAPER_RC):
    fig, axes = plt.subplots(2, 1, figsize=(1.6 * len(ORDER) + 2, 7.5), sharex=True)
    for ax, stat in zip(axes, ("test RMSE", "log sigma")):
        for i, s in enumerate(ORDER):
            for c, (st, colr) in enumerate(zip(chains[s], shades(SAMPLERS[s][0]))):
                ax.errorbar(i + (c - 1.5) * 0.17, st[stat].mean(), yerr=st[stat].std(), fmt="o", ms=8,
                            color=colr, ecolor=colr, elinewidth=2, capsize=0, mec="white", mew=0.7)
        if "nuts" in ORDER:
            ax.axvspan(ORDER.index("nuts") - 0.5, ORDER.index("nuts") + 0.5, color="0.92", zorder=0)
        ax.set_ylabel("test RMSE" if stat == "test RMSE" else r"$\log\sigma$")
    axes[-1].set_xticks(range(len(ORDER)), [SHORT[s] + (f"\n$w={W:g}$" if "sticky" in s else "") for s in ORDER])
    axes[0].set_title(r"large network, $10^7$ gradients per chain")
    fig.tight_layout()
    fig.savefig(PAPER_DIR / f"{DATASET}_large_chain_means.pdf", bbox_inches="tight")
    plt.show()

**Caption.** Chain means of the test RMSE (top) and $\log\sigma$ (bottom) for the large network on Boston, with $10^7$ gradient evaluations per chain and full-batch gradients. Each dot is one chain started from its own MAP, shaded from dark to light in MAP order, with a bar of one posterior standard deviation. NUTS (shaded) runs from the same four MAPs.

# Supplements

### Table, $\hat R$ for the large network

In [ ]:
lines = [r"\begin{tabular}{lcccc}", r"\toprule",
         r"Sampler & $w$ & $\hat R$ $\log\sigma$ & $\hat R$ train RMSE & $\hat R$ test RMSE \\", r"\midrule"]
for s in ORDER:
    r = [rhat(stat_array(s, k)) for k in ("log sigma", "train RMSE", "test RMSE")]
    wtex = f"{W:g}" if "sticky" in s else "--"
    lines.append(f"{SAMPLERS[s][1]} & {wtex} & {r[0]:.2f} & {r[1]:.2f} & {r[2]:.2f} \\\\")
lines += [r"\bottomrule", r"\end{tabular}"]
print("\n".join(lines))

**Caption.** Rank-normalised $\hat R$ over four chains started from different MAPs for the large network on Boston, with $10^7$ gradient evaluations per PDMP chain. NUTS pools four chains from each of the same four MAPs.